# 03 — Enhanced Visualizations: Fashion Neutrality & Economic Indicators

This capstone notebook merges the yearly runway color metrics extracted from 2,400 Vogue Runway images (2000–2023) with the economic panel assembled in notebook 01 and the Google Trends data from notebook 02.  
It produces seven visualization sections that together test whether fashion runway palettes become more neutral/muted during economic downturns.

**Color metrics source:** K-Means clustering in CIELab space — `mean_chroma` (lower = more muted/neutral) and `neutral_share` (fraction of palette classified as neutral).  
**Economic panel:** `/data/external/economic_panel_yearly.csv`  
**Google Trends:** `/data/external/google_trends_all.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ── Figure output directory ─────────────────────────────────────────────────
FIGURES_DIR = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', 'data', 'figures'
)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Brand palette ────────────────────────────────────────────────────────────
CHARCOAL = '#2b2b2b'
SAND     = '#c9b99a'
MIST     = '#d6d0c8'
BONE     = '#f5f2ee'
ACCENT   = '#8c7b68'

# ── Global matplotlib style ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':      'white',
    'axes.facecolor':        'white',
    'axes.edgecolor':        '#d6d0c8',
    'axes.labelcolor':       '#2b2b2b',
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.spines.left':      False,
    'axes.spines.bottom':    True,
    'xtick.color':           '#2b2b2b',
    'ytick.color':           '#2b2b2b',
    'xtick.labelsize':       9,
    'ytick.labelsize':       9,
    'axes.labelsize':        10,
    'axes.titlesize':        12,
    'axes.titleweight':      'bold',
    'axes.titlecolor':       '#2b2b2b',
    'grid.color':            '#d6d0c8',
    'grid.linewidth':         0.6,
    'font.family':           'serif',
    'text.color':            '#2b2b2b',
    'legend.frameon':        False,
    'legend.fontsize':        9,
})

# ── Recession bands ───────────────────────────────────────────────────────────
RECESSIONS = [(2001, 2001), (2007, 2009), (2020, 2020)]

def add_recession_bands(ax, recessions=RECESSIONS, alpha=0.15, color=MIST):
    """Shade recession periods on an axis."""
    for start, end in recessions:
        ax.axvspan(start - 0.5, end + 0.5, alpha=alpha, color=color, zorder=0)

def style_axis(ax, ylabel=None, title=None):
    """Apply consistent axis styling: y-grid, no left spine, no y-tick marks."""
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=3)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)

print('Setup complete. Figures will be saved to:', os.path.abspath(FIGURES_DIR))

## Data Loading & Preparation

In [ ]:
# ── Recreate yearly color metrics from original notebook outputs ─────────────
yearly_data = {
    'year': list(range(2000, 2024)),
    'mean_chroma': [
        16.71, 16.13, 14.75, 14.29, 14.25, 14.72, 13.72, 13.11, 13.28, 14.05,
        14.64, 14.48, 13.59, 14.32, 14.89, 15.54, 14.31, 15.97, 15.44, 15.06,
        13.84, 14.09, 14.37, 14.21,
    ],
    'neutral_share': [
        0.699, 0.702, 0.742, 0.747, 0.774, 0.751, 0.785, 0.803, 0.793, 0.769,
        0.750, 0.752, 0.784, 0.762, 0.747, 0.728, 0.760, 0.719, 0.730, 0.739,
        0.773, 0.763, 0.755, 0.757,
    ],
}
chroma_df = pd.DataFrame(yearly_data)

# ── Load economic panel ───────────────────────────────────────────────────────
DATA_DIR = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', 'data', 'external'
)
econ_df = pd.read_csv(os.path.join(DATA_DIR, 'economic_panel_yearly.csv'))
econ_df['year'] = econ_df['year'].astype(int)

# ── Merge ─────────────────────────────────────────────────────────────────────
df = chroma_df.merge(econ_df, on='year', how='left')
df = df.sort_values('year').reset_index(drop=True)

# ── Build lagged columns (used in Section 7) ─────────────────────────────────
df['consumer_sentiment_lag1'] = df['consumer_sentiment'].shift(1)
df['recession_lag1']          = df['recession_share'].shift(1)

print(f'Merged dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print('Columns:', list(df.columns))
df.head()

## Section 1: Multi-Indicator Recession Dashboard

Three time-series panels share the same x-axis so the reader can visually assess co-movement between runway chroma, consumer sentiment, and unemployment.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
fig.suptitle(
    'Fashion Neutrality and Economic Indicators, 2000\u20132023',
    fontsize=14, fontweight='bold', color=CHARCOAL, y=1.01
)

years = df['year'].values

# ── Panel 1: Mean Chroma ──────────────────────────────────────────────────────
ax0 = axes[0]
ax0.plot(years, df['mean_chroma'], color=ACCENT, lw=2, marker='o',
         markersize=4, markerfacecolor='white', markeredgecolor=ACCENT,
         label='Mean Chroma')
add_recession_bands(ax0)
style_axis(ax0, ylabel='Mean Chroma (CIELab)', title='Runway Colour Saturation')
ax0.set_xlim(years.min() - 0.5, years.max() + 0.5)
# Recession legend patch
rec_patch = Patch(facecolor=MIST, alpha=0.6, label='NBER Recession')
ax0.legend(handles=[
    plt.Line2D([0], [0], color=ACCENT, lw=2, label='Mean Chroma'),
    rec_patch
], loc='upper right')

# ── Panel 2: Consumer Sentiment ───────────────────────────────────────────────
ax1 = axes[1]
ax1.plot(years, df['consumer_sentiment'], color=SAND, lw=2, marker='s',
         markersize=4, markerfacecolor='white', markeredgecolor=SAND,
         label='Consumer Sentiment')
add_recession_bands(ax1)
style_axis(ax1, ylabel='UMCSENT Index', title='University of Michigan Consumer Sentiment')
ax1.legend(loc='upper right')

# ── Panel 3: Unemployment Rate ────────────────────────────────────────────────
ax2 = axes[2]
ax2.plot(years, df['unemployment_rate'], color=CHARCOAL, lw=2, marker='^',
         markersize=4, markerfacecolor='white', markeredgecolor=CHARCOAL,
         label='Unemployment Rate')
add_recession_bands(ax2)
style_axis(ax2, ylabel='Unemployment Rate (%)', title='U.S. Unemployment Rate')
ax2.set_xlabel('Year')
ax2.set_xticks(range(2000, 2024, 2))
ax2.legend(loc='upper right')

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section1_multi_indicator_dashboard.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: section1_multi_indicator_dashboard.png')

## Section 2: Chroma vs. Consumer Sentiment

Tests the hypothesis that lower consumer confidence predicts more neutral/muted runway palettes.  
Points are coloured by year using the `copper_r` colormap.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Drop rows with missing sentiment
scatter_df = df.dropna(subset=['consumer_sentiment', 'mean_chroma']).copy()

cmap = matplotlib.colormaps.get_cmap('copper_r')
norm = matplotlib.colors.Normalize(
    vmin=scatter_df['year'].min(), vmax=scatter_df['year'].max()
)
colors = [cmap(norm(y)) for y in scatter_df['year']]

sc = ax.scatter(
    scatter_df['consumer_sentiment'], scatter_df['mean_chroma'],
    c=scatter_df['year'], cmap='copper_r',
    s=60, zorder=3, edgecolors='white', linewidths=0.5
)

# Annotate year labels
for _, row in scatter_df.iterrows():
    ax.annotate(
        str(int(row['year'])),
        (row['consumer_sentiment'], row['mean_chroma']),
        fontsize=7, color=CHARCOAL, alpha=0.7,
        xytext=(3, 3), textcoords='offset points'
    )

# Regression line
x_vals = scatter_df['consumer_sentiment'].values
y_vals = scatter_df['mean_chroma'].values
slope, intercept, r_val, p_val, se = stats.linregress(x_vals, y_vals)
x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
ax.plot(x_line, slope * x_line + intercept, color=ACCENT, lw=1.8,
        linestyle='--', label=f'OLS fit  R²={r_val**2:.3f}, p={p_val:.3f}')

cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Year', fontsize=9)
cbar.ax.tick_params(labelsize=8)

style_axis(ax,
    ylabel='Mean Chroma (CIELab)',
    title='Runway Chroma vs. Consumer Sentiment'
)
ax.set_xlabel('University of Michigan Consumer Sentiment (UMCSENT)')
ax.legend(loc='lower right')

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section2_chroma_vs_sentiment.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print(f'R² = {r_val**2:.4f},  slope = {slope:.4f},  p = {p_val:.4f}')
print('Saved: section2_chroma_vs_sentiment.png')

## Section 3: Chroma vs. Unemployment Rate

Enhanced version of the original project's scatter.  
Points are coloured by `recession_share` — darker = more months in recession.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

scatter_df2 = df.dropna(subset=['unemployment_rate', 'mean_chroma']).copy()

sc2 = ax.scatter(
    scatter_df2['unemployment_rate'], scatter_df2['mean_chroma'],
    c=scatter_df2['recession_share'], cmap='YlOrBr',
    s=65, zorder=3, edgecolors='white', linewidths=0.5, vmin=0, vmax=1
)

for _, row in scatter_df2.iterrows():
    ax.annotate(
        str(int(row['year'])),
        (row['unemployment_rate'], row['mean_chroma']),
        fontsize=7, color=CHARCOAL, alpha=0.7,
        xytext=(3, 3), textcoords='offset points'
    )

x_vals2 = scatter_df2['unemployment_rate'].values
y_vals2 = scatter_df2['mean_chroma'].values
slope2, intercept2, r_val2, p_val2, se2 = stats.linregress(x_vals2, y_vals2)
x_line2 = np.linspace(x_vals2.min(), x_vals2.max(), 200)
ax.plot(x_line2, slope2 * x_line2 + intercept2,
        color=ACCENT, lw=1.8, linestyle='--',
        label=f'OLS fit  R²={r_val2**2:.3f}, p={p_val2:.3f}')

cbar2 = fig.colorbar(sc2, ax=ax, pad=0.01)
cbar2.set_label('Recession Share\n(fraction of year in recession)', fontsize=8)
cbar2.ax.tick_params(labelsize=8)

style_axis(ax,
    ylabel='Mean Chroma (CIELab)',
    title='Runway Chroma vs. Unemployment Rate'
)
ax.set_xlabel('Annual Unemployment Rate (%)')
ax.legend(loc='upper right')

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section3_chroma_vs_unemployment.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print(f'R² = {r_val2**2:.4f},  slope = {slope2:.4f},  p = {p_val2:.4f}')
print('Saved: section3_chroma_vs_unemployment.png')

## Section 4: Correlation Heatmap — Chroma × All Indicators

Pearson correlation matrix across all key variables. Diverging palette highlights positive and negative relationships.

In [ ]:
corr_cols = [
    'mean_chroma', 'neutral_share',
    'unemployment_rate', 'consumer_sentiment',
    'gdp_growth', 'sp500_yoy_change', 'recession_share'
]
corr_labels = [
    'Mean Chroma', 'Neutral Share',
    'Unemployment', 'Consumer Sentiment',
    'GDP Growth', 'S&P 500 YoY', 'Recession Share'
]

corr_df = df[corr_cols].rename(columns=dict(zip(corr_cols, corr_labels)))
corr_matrix = corr_df.corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 7))

# Custom diverging cmap toned to project palette (teal endpoint replaced with MIST)
cmap_div = sns.diverging_palette(20, 220, s=40, l=65, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap_div,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, linecolor=MIST,
    square=True, ax=ax,
    cbar_kws={'shrink': 0.75, 'label': 'Pearson r'}
)

ax.set_title('Correlation Heatmap: Chroma × Economic Indicators',
             fontsize=12, fontweight='bold', color=CHARCOAL, pad=12)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

# Remove spines on heatmap
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section4_correlation_heatmap.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: section4_correlation_heatmap.png')
print('\nTop correlations with mean_chroma:')
print(corr_matrix['Mean Chroma'].sort_values().to_string())

## Section 5: Google Trends Composite vs. Chroma (2004–2023)

Dual-axis plot: does public search interest in neutral/quiet-fashion keywords track actual runway neutrality?

In [ ]:
gt_raw = pd.read_csv(os.path.join(DATA_DIR, 'google_trends_all.csv'), parse_dates=['date'])

keyword_cols = [c for c in gt_raw.columns if c != 'date']
gt_raw['composite'] = gt_raw[keyword_cols].mean(axis=1)

gt_raw['year'] = gt_raw['date'].dt.year
gt_yearly = gt_raw.groupby('year')['composite'].mean().reset_index()
gt_yearly.columns = ['year', 'gt_composite']

# Merge with chroma (overlap 2004-2023)
trends_df = chroma_df.merge(gt_yearly, on='year', how='inner')
print(f'Overlapping years: {trends_df["year"].min()} – {trends_df["year"].max()}')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

t_years = trends_df['year'].values

ln1 = ax1.plot(t_years, trends_df['mean_chroma'],
               color=ACCENT, lw=2, marker='o',
               markersize=4, markerfacecolor='white', markeredgecolor=ACCENT,
               label='Mean Chroma (left axis)')

ln2 = ax2.plot(t_years, trends_df['gt_composite'],
               color='#5b7fa6', lw=2, linestyle='-.', marker='s',
               markersize=4, markerfacecolor='white', markeredgecolor='#5b7fa6',
               label='GT Composite Index (right axis)')

# Add recession shading on primary axis
for start, end in RECESSIONS:
    if start >= trends_df['year'].min():
        ax1.axvspan(start - 0.5, end + 0.5, alpha=0.15, color=MIST, zorder=0)

# Style primary axis
ax1.yaxis.grid(True)
ax1.set_axisbelow(True)
ax1.tick_params(axis='y', length=0)
ax1.tick_params(axis='x', length=3)
ax1.set_ylabel('Mean Chroma (CIELab)', color=ACCENT)
ax1.tick_params(axis='y', labelcolor=ACCENT)
ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_visible(False)

# Style secondary axis
ax2.tick_params(axis='y', length=0)
ax2.set_ylabel('Google Trends Composite (0–100)', color='#5b7fa6')
ax2.tick_params(axis='y', labelcolor='#5b7fa6')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color('#d6d0c8')

ax1.set_xlabel('Year')
ax1.set_xlim(t_years.min() - 0.5, t_years.max() + 0.5)
ax1.set_xticks(range(int(t_years.min()), int(t_years.max()) + 1, 2))

# Combined legend
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left')

ax1.set_title(
    'Google Trends Composite (Quiet-Fashion Keywords) vs. Runway Chroma',
    fontsize=12, fontweight='bold', color=CHARCOAL
)

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section5_trends_vs_chroma.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: section5_trends_vs_chroma.png')

# Correlation between the two series
r_gt, p_gt = stats.pearsonr(trends_df['mean_chroma'], trends_df['gt_composite'])
print(f'Pearson r (chroma vs GT composite): {r_gt:.4f}  (p={p_gt:.4f})')

## Section 6: Year-over-Year Change Analysis

Tests whether *changes* in economic conditions predict *changes* in runway neutrality — a more dynamic framing than levels-only analysis.

In [ ]:
# ── Compute YoY changes ───────────────────────────────────────────────────────
df['chroma_change']     = df['mean_chroma'].diff()
df['unemp_change']      = df['unemployment_rate'].diff()
df['sentiment_change']  = df['consumer_sentiment'].diff()

yoy_df = df.dropna(subset=['chroma_change', 'unemp_change', 'sentiment_change']).copy()
print(f'YoY dataset: {yoy_df.shape[0]} observations')

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(yoy_df))
years_yoy = yoy_df['year'].values

# Normalise each series to comparable scale for visual comparison
def zscore(s):
    return (s - s.mean()) / s.std()

ax.plot(years_yoy, zscore(yoy_df['chroma_change']),
        color=ACCENT, lw=2, marker='o', markersize=4,
        markerfacecolor='white', markeredgecolor=ACCENT,
        label='Δ Mean Chroma (z-scored)')

ax.plot(years_yoy, zscore(yoy_df['unemp_change']),
        color=CHARCOAL, lw=1.5, linestyle='--', marker='^', markersize=4,
        markerfacecolor='white', markeredgecolor=CHARCOAL,
        label='Δ Unemployment Rate (z-scored)')

ax.plot(years_yoy, zscore(yoy_df['sentiment_change']),
        color=SAND, lw=1.5, linestyle=':', marker='s', markersize=4,
        markerfacecolor='white', markeredgecolor=SAND,
        label='Δ Consumer Sentiment (z-scored)')

ax.axhline(0, color=MIST, lw=1)
add_recession_bands(ax)

style_axis(ax,
    ylabel='Z-scored Year-over-Year Change',
    title='Year-over-Year Changes in Chroma and Economic Indicators'
)
ax.set_xlabel('Year')
ax.set_xlim(years_yoy.min() - 0.5, years_yoy.max() + 0.5)
ax.set_xticks(range(int(years_yoy.min()), int(years_yoy.max()) + 1, 2))
ax.legend(loc='upper right')

plt.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, 'section6_yoy_changes.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: section6_yoy_changes.png')

In [ ]:
# ── OLS Regression: Δ chroma ~ Δ unemployment + Δ sentiment ─────────────────
print('=' * 60)
print('OLS: chroma_change ~ unemployment_change + sentiment_change')
print('=' * 60)

yoy_clean = yoy_df.dropna(subset=['chroma_change', 'unemp_change', 'sentiment_change'])
X_yoy = sm.add_constant(yoy_clean[['unemp_change', 'sentiment_change']])
y_yoy = yoy_clean['chroma_change']
model_yoy = sm.OLS(y_yoy, X_yoy).fit()
print(model_yoy.summary())

## Section 7: Enhanced Regression Models

Four OLS specifications, from the original project's baseline (Model A) to a lagged model (Model D).  
A summary comparison table is printed after the individual model outputs.

In [ ]:
# ── Model A: original spec ────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('MODEL A: mean_chroma ~ unemployment_rate + recession_share')
print('=' * 60)
df_a = df.dropna(subset=['mean_chroma', 'unemployment_rate', 'recession_share'])
X_a  = sm.add_constant(df_a[['unemployment_rate', 'recession_share']])
y_a  = df_a['mean_chroma']
model_a = sm.OLS(y_a, X_a).fit()
print(model_a.summary())

# ── Model B: sentiment spec ────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('MODEL B: mean_chroma ~ consumer_sentiment + recession_share')
print('=' * 60)
df_b = df.dropna(subset=['mean_chroma', 'consumer_sentiment', 'recession_share'])
X_b  = sm.add_constant(df_b[['consumer_sentiment', 'recession_share']])
y_b  = df_b['mean_chroma']
model_b = sm.OLS(y_b, X_b).fit()
print(model_b.summary())

# ── Model C: kitchen sink ──────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('MODEL C: mean_chroma ~ unemployment_rate + consumer_sentiment + recession_share + year')
print('=' * 60)
df_c = df.dropna(subset=['mean_chroma', 'unemployment_rate', 'consumer_sentiment', 'recession_share', 'year'])
X_c  = sm.add_constant(df_c[['unemployment_rate', 'consumer_sentiment', 'recession_share', 'year']])
y_c  = df_c['mean_chroma']
model_c = sm.OLS(y_c, X_c).fit()
print(model_c.summary())

# ── Model D: lagged spec ───────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('MODEL D: mean_chroma ~ unemployment_rate + consumer_sentiment_lag1 + recession_lag1 + year')
print('=' * 60)
df_d = df.dropna(subset=['mean_chroma', 'unemployment_rate', 'consumer_sentiment_lag1', 'recession_lag1', 'year'])
X_d  = sm.add_constant(df_d[['unemployment_rate', 'consumer_sentiment_lag1', 'recession_lag1', 'year']])
y_d  = df_d['mean_chroma']
model_d = sm.OLS(y_d, X_d).fit()
print(model_d.summary())

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('MODEL COMPARISON SUMMARY')
print('=' * 70)

models   = [model_a, model_b, model_c, model_d]
labels   = ['A (Baseline)', 'B (Sentiment)', 'C (Kitchen Sink)', 'D (Lagged)']
specs    = [
    'unemp + rec_share',
    'sentiment + rec_share',
    'unemp + sentiment + rec + year',
    'unemp + sentiment_L1 + rec_L1 + year',
]

header = f"{'Model':<20} {'Spec':<38} {'R²':>6} {'Adj R²':>8} {'AIC':>8} {'N':>4}"
print(header)
print('-' * 86)
for lbl, spec, m in zip(labels, specs, models):
    print(f"{lbl:<20} {spec:<38} {m.rsquared:>6.3f} {m.rsquared_adj:>8.3f} {m.aic:>8.2f} {int(m.nobs):>4}")

print('\n\nCoefficient Detail (statistically significant at p<0.10 marked *)')
print('-' * 86)
for lbl, m in zip(labels, models):
    print(f"\n  {lbl}")
    for var in m.params.index:
        coef  = m.params[var]
        pval  = m.pvalues[var]
        star  = '*' if pval < 0.10 else ' '
        print(f"    {var:<35} coef={coef:>8.4f}  p={pval:.4f} {star}")

## Summary & Interpretation

| Section | Key Finding |
|---------|-------------|
| 1 | The triple dashboard makes the co-movement visually legible — chroma dips cluster near the 2008–09 and 2020 recession bands. |
| 2 | Chroma–Sentiment scatter: lower confidence generally pairs with lower chroma, though the relationship is modest. |
| 3 | Chroma–Unemployment scatter: years with elevated unemployment (2009–2011) show some of the lowest chroma values. |
| 4 | Heatmap reveals `consumer_sentiment` and `mean_chroma` share a positive correlation, while `unemployment` and `recession_share` correlate negatively with chroma. |
| 5 | Google Trends composite (quiet-fashion keywords) shows near-zero engagement until ~2020, then spikes sharply — partially tracking the 2020 chroma dip and subsequent plateau. |
| 6 | YoY change OLS: year-to-year deteriorations in sentiment and increases in unemployment are associated with negative Δchroma, but individual predictors remain noisy on an annual basis. |
| 7 | Model B (sentiment) outperforms Model A (unemployment) in R². Model C raises R² further but risks overfitting given n=24. Model D (lagged) provides a causal interpretation — prior-year sentiment and recession exposure predict current-year chroma. |

**Overall verdict:** There is consistent but modest statistical evidence that runway colour saturation falls during and immediately after economic downturns. Consumer sentiment is a somewhat stronger contemporaneous predictor than the unemployment rate alone.